In [32]:
#作业一： 写一个wrap_model_call中间件，根据条件动态替换模型。
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.messages import HumanMessage

local_model  = init_chat_model(model="ollama:qwen3.5:0.8b")
remote_model = init_chat_model(model="ollama:qwen3.6:latest",base_url="http://192.168.8.21:11434")

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """对话一次后，就切换大模型"""
    message_count = len(request.state["messages"])

    if message_count > 2:
        # Use an advanced model for longer conversations
        print("当前是使用的服务器的模型👀")
        model = remote_model
    else:
        print("当前是使用的本地的模型👀")
        model = local_model

    return handler(request.override(model=model))

agent = create_agent(
    model=local_model,  # Default model
    middleware=[dynamic_model_selection]
)

messages = []

print("第一次：")
messages.append(HumanMessage("55*55 = ?"))
response01 = agent.invoke({"messages": messages})
messages = response01["messages"]
print(response01["messages"][-1].content)

print("第二次：")
messages.append(HumanMessage("(100+200)*90=?"))
response02 = agent.invoke({"messages": messages})
messages = response02["messages"]
print(response02["messages"][-1].content)


print("第三次：")
messages.append(HumanMessage("五一怎么玩，简短攻略"))
response03 = agent.invoke({"messages": messages})
messages = response03["messages"]
print(response03["messages"][-1].content)



第一次：
当前是使用的本地的模型👀


KeyboardInterrupt: 

In [36]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

# =========================
# 1️⃣ 初始化模型
# =========================
main_model = init_chat_model(
    model="ollama:qwen3.6:latest",
    base_url="http://192.168.8.21:11434"
)

summary_model = init_chat_model(
    model="ollama:qwen3.6:latest",
    base_url="http://192.168.8.21:11434"
)

# =========================
# 2️⃣ 工具函数
# =========================
def format_messages(messages):
    return "\n".join([f"{m.type}: {m.content}" for m in messages])

# =========================
# 3️⃣ 摘要函数
# =========================
def summarize(messages):
    system_prompt = """请总结以下对话，保留：
1. 用户核心问题
2. 已有结论
3. 重要上下文
不超过100字"""

    response = summary_model.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=format_messages(messages))
    ])

    return response.content

# =========================
# 4️⃣ 中间件（带调试打印）
# =========================
@wrap_model_call
def summary_middleware(request: ModelRequest, handler) -> ModelResponse:
    
    messages = request.messages

    MAX_MESSAGES = 6
    KEEP_LAST = 3

    print("\n====== 📦 Middleware收到的 messages ======")
    print(f"长度: {len(messages)}")

    if len(messages) > MAX_MESSAGES:
        print("\n⚡ 触发上下文压缩")

        old_messages = messages[:-KEEP_LAST]
        recent_messages = messages[-KEEP_LAST:]

        print(f"\n📊 压缩前长度: {len(messages)}")

        # 打印被压缩部分
        print("\n------ 🧠 被摘要的历史 ------")
        for m in old_messages:
            print(f"[{m.type}] {m.content}")

        summary_text = summarize(old_messages)

        print("\n------ 🧾 生成的摘要 ------")
        print(summary_text)

        new_messages = [
            SystemMessage(content="以下是之前对话的摘要："),
            SystemMessage(content=summary_text),
            *recent_messages
        ]

        request.messages = new_messages

        print(f"\n📊 压缩后长度: {len(request.messages)}")

        print("\n====== 📉 压缩后 messages ======")
        for i, m in enumerate(request.messages):
            print(f"{i}. [{m.type}] {m.content}")

        print("================================\n")

    return handler(request)

# =========================
# 5️⃣ 创建 Agent
# =========================
agent = create_agent(
    model=main_model,
    middleware=[summary_middleware]
)

# =========================
# 6️⃣ 测试（重点）
# =========================
def test():
    messages = []

    question = [
        "1+1=？",
        "1+2=？",
        "1+6=？",
        "1+9=？",
        "99+100=？",
        "101+99=？",
        "109+99=？",
        "199+99=？",
        "109*99=？",
        "帮我总结一下刚才的内容"
    ]

    for i, q in enumerate(question):
        user_input = f"第{i+1}问: {q}"

        messages.append(HumanMessage(content=user_input))

        print("\n==============================")
        print(f"👤 用户输入: {user_input}")
        print(f"📊 外部messages长度: {len(messages)}")

        response = agent.invoke({
            "messages": messages
        })

        ai_msg = response["messages"][-1]
        messages.append(ai_msg)

        print(f"🤖 AI回复: {ai_msg.content}")
        print(f"📊 外部messages长度(追加后): {len(messages)}")

    print("\n🔥 最终外部 messages 总长度:", len(messages))


if __name__ == "__main__":
    test()


👤 用户输入: 第1问: 1+1=？
📊 外部messages长度: 1

====== 📦 Middleware收到的 messages ======
长度: 1
🤖 AI回复: 1+1=2。
📊 外部messages长度(追加后): 2

👤 用户输入: 第2问: 1+2=？
📊 外部messages长度: 3

====== 📦 Middleware收到的 messages ======
长度: 3
🤖 AI回复: 1+2=3。
📊 外部messages长度(追加后): 4

👤 用户输入: 第3问: 1+6=？
📊 外部messages长度: 5

====== 📦 Middleware收到的 messages ======
长度: 5
🤖 AI回复: 1+6=7。
📊 外部messages长度(追加后): 6

👤 用户输入: 第4问: 1+9=？
📊 外部messages长度: 7

====== 📦 Middleware收到的 messages ======
长度: 7

⚡ 触发上下文压缩

📊 压缩前长度: 7

------ 🧠 被摘要的历史 ------
[human] 第1问: 1+1=？
[ai] 1+1=2。
[human] 第2问: 1+2=？
[ai] 1+2=3。

------ 🧾 生成的摘要 ------
用户核心问题：计算1+1与1+2的结果。
已有结论：1+1=2，1+2=3。
重要上下文：用户连续提问两道基础加法题，AI逐一给出准确答案的简单问答交互。

📊 压缩后长度: 5

====== 📉 压缩后 messages ======
0. [system] 以下是之前对话的摘要：
1. [system] 用户核心问题：计算1+1与1+2的结果。
已有结论：1+1=2，1+2=3。
重要上下文：用户连续提问两道基础加法题，AI逐一给出准确答案的简单问答交互。
2. [human] 第3问: 1+6=？
3. [ai] 1+6=7。
4. [human] 第4问: 1+9=？



/var/folders/mb/f_7rzxgd24j1s22_0c4kt23m0000gn/T/ipykernel_31824/1410219332.py:80: DeprecationWarning: Direct attribute assignment to ModelRequest.messages is deprecated. Use request.override(messages=...) instead to create a new request with the modified attribute.
  request.messages = new_messages


🤖 AI回复: 1+9=10。
📊 外部messages长度(追加后): 8

👤 用户输入: 第5问: 99+100=？
📊 外部messages长度: 9

====== 📦 Middleware收到的 messages ======
长度: 9

⚡ 触发上下文压缩

📊 压缩前长度: 9

------ 🧠 被摘要的历史 ------
[human] 第1问: 1+1=？
[ai] 1+1=2。
[human] 第2问: 1+2=？
[ai] 1+2=3。
[human] 第3问: 1+6=？
[ai] 1+6=7。

------ 🧾 生成的摘要 ------
用户依次提问三道基础加法（1+1、1+2、1+6），AI均给出正确答案（2、3、7）。对话属连续算术问答，核心内容为检验基础计算能力。

📊 压缩后长度: 5

====== 📉 压缩后 messages ======
0. [system] 以下是之前对话的摘要：
1. [system] 用户依次提问三道基础加法（1+1、1+2、1+6），AI均给出正确答案（2、3、7）。对话属连续算术问答，核心内容为检验基础计算能力。
2. [human] 第4问: 1+9=？
3. [ai] 1+9=10。
4. [human] 第5问: 99+100=？



/var/folders/mb/f_7rzxgd24j1s22_0c4kt23m0000gn/T/ipykernel_31824/1410219332.py:80: DeprecationWarning: Direct attribute assignment to ModelRequest.messages is deprecated. Use request.override(messages=...) instead to create a new request with the modified attribute.
  request.messages = new_messages


🤖 AI回复: 99+100=199。
📊 外部messages长度(追加后): 10

👤 用户输入: 第6问: 101+99=？
📊 外部messages长度: 11

====== 📦 Middleware收到的 messages ======
长度: 11

⚡ 触发上下文压缩

📊 压缩前长度: 11

------ 🧠 被摘要的历史 ------
[human] 第1问: 1+1=？
[ai] 1+1=2。
[human] 第2问: 1+2=？
[ai] 1+2=3。
[human] 第3问: 1+6=？
[ai] 1+6=7。
[human] 第4问: 1+9=？
[ai] 1+9=10。

------ 🧾 生成的摘要 ------
用户核心问题：依次提问四道基础加法题（1+1至1+9）。
已有结论：AI已准确作答（结果分别为2、3、7、10）。
重要上下文：对话为连续的四轮数学问答，无其他背景信息。

📊 压缩后长度: 5

====== 📉 压缩后 messages ======
0. [system] 以下是之前对话的摘要：
1. [system] 用户核心问题：依次提问四道基础加法题（1+1至1+9）。
已有结论：AI已准确作答（结果分别为2、3、7、10）。
重要上下文：对话为连续的四轮数学问答，无其他背景信息。
2. [human] 第5问: 99+100=？
3. [ai] 99+100=199。
4. [human] 第6问: 101+99=？



...

🤖 AI回复: 199+99=298。
📊 外部messages长度(追加后): 16

👤 用户输入: 第9问: 109*99=？
📊 外部messages长度: 17

====== 📦 Middleware收到的 messages ======
长度: 17

⚡ 触发上下文压缩

📊 压缩前长度: 17

------ 🧠 被摘要的历史 ------
[human] 第1问: 1+1=？
[ai] 1+1=2。
[human] 第2问: 1+2=？
[ai] 1+2=3。
[human] 第3问: 1+6=？
[ai] 1+6=7。
[human] 第4问: 1+9=？
[ai] 1+9=10。
[human] 第5问: 99+100=？
[ai] 99+100=199。
[human] 第6问: 101+99=？
[ai] 101+99=200。
[human] 第7问: 109+99=？
[ai] 109+99=208。

------ 🧾 生成的摘要 ------
1.核心问题：用户连续提出7道基础加法题。
2.已有结论：AI均给出准确答案（结果分别为2至208）。
3.重要上下文：纯数学运算问答，无额外背景或复杂条件，仅用于核对基础算术结果。

📊 压缩后长度: 5

====== 📉 压缩后 messages ======
0. [system] 以下是之前对话的摘要：
1. [system] 1.核心问题：用户连续提出7道基础加法题。
2.已有结论：AI均给出准确答案（结果分别为2至208）。
3.重要上下文：纯数学运算问答，无额外背景或复杂条件，仅用于核对基础算术结果。
2. [human] 第8问: 199+99=？
3. [ai] 199+99=298。
4. [human] 第9问: 109*99=？



/var/folders/mb/f_7rzxgd24j1s22_0c4kt23m0000gn/T/ipykernel_31824/1410219332.py:80: DeprecationWarning: Direct attribute assignment to ModelRequest.messages is deprecated. Use request.override(messages=...) instead to create a new request with the modified attribute.
  request.messages = new_messages


🤖 AI回复: 109*99=10791。
📊 外部messages长度(追加后): 18

👤 用户输入: 第10问: 帮我总结一下刚才的内容
📊 外部messages长度: 19

====== 📦 Middleware收到的 messages ======
长度: 19

⚡ 触发上下文压缩

📊 压缩前长度: 19

------ 🧠 被摘要的历史 ------
[human] 第1问: 1+1=？
[ai] 1+1=2。
[human] 第2问: 1+2=？
[ai] 1+2=3。
[human] 第3问: 1+6=？
[ai] 1+6=7。
[human] 第4问: 1+9=？
[ai] 1+9=10。
[human] 第5问: 99+100=？
[ai] 99+100=199。
[human] 第6问: 101+99=？
[ai] 101+99=200。
[human] 第7问: 109+99=？
[ai] 109+99=208。
[human] 第8问: 199+99=？
[ai] 199+99=298。

------ 🧾 生成的摘要 ------
用户核心问题为连续求解8道加法题。已有结论为AI依次给出答案：2、3、7、10、199、200、208、298。重要上下文为逐题一问一答的算术计算过程。

📊 压缩后长度: 5

====== 📉 压缩后 messages ======
0. [system] 以下是之前对话的摘要：
1. [system] 用户核心问题为连续求解8道加法题。已有结论为AI依次给出答案：2、3、7、10、199、200、208、298。重要上下文为逐题一问一答的算术计算过程。
2. [human] 第9问: 109*99=？
3. [ai] 109*99=10791。
4. [human] 第10问: 帮我总结一下刚才的内容



/var/folders/mb/f_7rzxgd24j1s22_0c4kt23m0000gn/T/ipykernel_31824/1410219332.py:80: DeprecationWarning: Direct attribute assignment to ModelRequest.messages is deprecated. Use request.override(messages=...) instead to create a new request with the modified attribute.
  request.messages = new_messages


🤖 AI回复: 本次对话以连续解答数学题为主线，整体过程如下：

1. **前8问（加法运算）**：采用逐题一问一答的节奏，依次求解并得出答案：`2、3、7、10、199、200、208、298`。
2. **第9问（乘法运算）**：题目为 `109×99`，计算结果为 `10791`。
3. **第10问（当前请求）**：您要求对刚才的解题过程与内容进行整体总结。

整体来看，对话保持了清晰的递进结构，题型从前期的基础加法逐步过渡到两位数乘法，问答节奏稳定，结果均已准确给出。如需继续新增题目或调整题型，可随时提出。
📊 外部messages长度(追加后): 20

🔥 最终外部 messages 总长度: 20
